# L4 GRPO Demo — Bioresearch (Colab Pro / single-target L4, Qwen2.5-3B, All 14 Tasks)

<a href="https://colab.research.google.com/github/<GITHUB_OWNER>/bioresearch/blob/main/notebooks/train_grpo_l4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

End-to-end GRPO training run, **hard-targeted at a Colab Pro L4 (22.5 GB, Ada sm_89, bf16 native)**, that trains and evaluates **all 14 bioresearch tasks** in a single sitting. Sister notebooks:

- [`train_grpo_t4.ipynb`](train_grpo_t4.ipynb) — Colab Free T4 (16 GB, Turing, fp16 only, Qwen 1.5B)
- [`train_grpo_colab_pro.ipynb`](train_grpo_colab_pro.ipynb) — Colab Pro auto-detect (Qwen 7B on either A100 or L4)
- [`train_grpo_l40s.ipynb`](train_grpo_l40s.ipynb) — single L40s 48 GB (Qwen 7B)
- [`train_grpo_a100.ipynb`](train_grpo_a100.ipynb) — single A100 (Qwen 7B)

Same helpers, same reward stack, same artifacts — retuned for the 22.5 GB / bf16-native / sm_89 reality of L4 with **Qwen2.5-3B** as the model size sweet spot.

Engineered around the hackathon's two weakest-typically axes:

- **Observable evidence of training progress (20%):**
  - **Live Trackio dashboard** that judges can watch during the demo (auto-syncs to a HF Space if `HF_TOKEN` + `HF_SPACE_ID` are set, so it persists after the runtime dies).
  - **Baseline-vs-trained eval block** that runs the *exact same* held-out `task_id`s before training and after training, then renders a side-by-side table + sample rollout transcripts.
- **Reward coherent + meaningful inference improvement (10%):**
  - **Reward-distribution diagnostic** sampled before training so we can see the reward signal has variance (not stuck at 0 or 1) — without this the GRPO advantage collapses.
  - **`LabShapingCallback`** decomposes lab-task reward into *process* (per-step shaping) and *terminal* (final outcome) so the curves show whether training is learning to *think* better or just to *guess* better.
  - **`save_steps=30`** writes the LoRA every ~15 min so a mid-run disconnect does not lose more than one checkpoint.

All heavy logic lives in [`training_core.py`](../training_core.py) and [`training_a100.py`](../training_a100.py) — both are GPU-agnostic, so we reuse them verbatim. This notebook is a thin driver.

**Hardware target:** NVIDIA L4 22.5 GB (Ada Lovelace, sm_89, ~300 GB/s GDDR6, **bf16 native**, **flash-attn 2 supported but not required**).

---

## Why Qwen2.5-3B (and not 1.5B / 7B) on a 22.5 GB Ada card

We hard-pin Qwen2.5-3B-Instruct (4-bit) — the model-size sweet spot for L4:

- **vs T4's 1.5B** (`train_grpo_t4.ipynb`): 3B has roughly 2x the parameter count, which closes most of the knowledge-ceiling gap on the two stubborn tasks (`protein_function`, `kegg_pathway_reasoning`) where GRPO can only optimize over what the base model already knows. Expect visible green deltas on those tasks where T4's run shows flat / zero deltas.
- **vs Colab Pro auto-detect's 7B** (`train_grpo_colab_pro.ipynb`): 7B on L4 works (the `colab_pro` notebook proves it), but it sits at ~85% VRAM utilization with `num_generations=8` and is a known dance with Unsloth's `matmul_lora` dtype kernel under gradient checkpointing. 3B leaves us ~12-14 GB of headroom — comfortable for `num_generations=8` plus deep lab rollouts (`TRAIN_LAB_MAX_STEPS=4`) — and sidesteps the dtype edge cases entirely.
- **bf16 native is a real upgrade over T4's fp16.** Same speed, ~half the gradient-noise variance on small-batch GRPO updates. We hardcode `bf16=True` and skip the T4 notebook's `fp16=True` forcing.

**Net Colab Pro L4 wall-clock for the full 14-task demo run:** ~55-70 min train + ~10 min eval = **~65-80 min total**.

**Why this is the safest L4 notebook in the family.** Today's debugging on the 7B `colab_pro` notebook surfaced a chain of Unsloth + PEFT + TRL GRPO + 4-bit + gradient-checkpointing dtype interactions that crash inside `matmul_lora`. **Every fix from that debugging is baked in here from the start:**

1. `dtype=torch.bfloat16` is passed explicitly to `FastLanguageModel.from_pretrained` (Cell 14)
2. `use_gradient_checkpointing=True` (HF native) instead of `"unsloth"` so recompute respects autocast (Cell 14)
3. `model.generation_config.max_length = None` to silence the `~1300x` HF warning spam (Cell 14)
4. Trainable params are cast back to bf16 *after* `GRPOTrainer.__init__` (Cell 20) — PEFT's `autocast_adapter_dtype=True` re-promotes them, this re-pins them
5. `trainer.args.bf16/fp16` is reconciled with the actual base dtype (Cell 20)

If you want max quality at the cost of more wall-clock + dtype risk, switch to `train_grpo_colab_pro.ipynb` — it auto-detects A100/L4 and runs 7B. If you want max compatibility on a free runtime, use `train_grpo_t4.ipynb`. This notebook is the **L4-pinned middle ground** — no auto-detect overhead, no model-size guessing, no dtype gremlins.

**Hackathon checklist** (all artifacts produced by this notebook):

- [x] Live Trackio dashboard (Cell 8) — `bioresearch-grpo-l4` project, optional HF Space sync
- [x] Reward-distribution diagnostic + boxplot (Cell 18)
- [x] Baseline rollouts on paired task_ids (Cell 16)
- [x] GRPO training run with mid-run LoRA checkpoints every 30 steps (Cells 20-22)
- [x] Trained rollouts on the same paired task_ids (Cell 25)
- [x] Before-vs-after table + bar chart (Cell 27)
- [x] Sample rollout transcripts side-by-side (Cell 29)
- [x] Per-task reward curves (Cell 31) — static fallback in case Trackio is offline
- [x] Saved LoRA + optional HF Hub push (Cell 33) — ~75 MB


## 1 · Install dependencies

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# L4 = Ada Lovelace (sm_89) = bf16 native, fp16 supported, flash-attn 2 supported.
# Unsloth's xformers fallback works fine; we don't bother with flash-attn 2 because
# it's not the bottleneck at num_generations=8 + max_completion=384 (lab rollouts
# dominate wall-clock, and they're sequential autoregressive generates either way).
# xformers<0.0.28 ships sm_89 wheels in the same release as sm_75 (T4) and sm_80
# (A100), so the pin matches train_grpo_t4.ipynb / train_grpo_l40s.ipynb verbatim.
#
# trl: GRPOConfig/GRPOTrainer landed in 0.13.0 (Dec 2024); processing_class= and
# vllm_mode= matured in 0.15+. We pin >=0.15,<1.0 so pip resolves to ~0.18.x
# (the last mature 0.x line) and avoids the 1.x reorg.
!pip install -q --no-deps "xformers<0.0.28" "trl>=0.15,<1.0" peft accelerate bitsandbytes
!pip install -q httpx fastapi uvicorn pydantic pandas matplotlib datasets scipy
!pip install -q trackio  # live dashboard; the pipeline still runs without it

# vLLM is intentionally NOT installed on Colab L4 (same reasoning as T4):
#   - --no-deps install drops vLLM module files without its runtime deps, which
#     triggers a `ModuleNotFoundError: cv2` deep inside Unsloth's startup hook
#     `fix_vllm_guided_decoding_params()` -> `import vllm`.
#   - Full install (~500 MB of wheels including ray, opencv-python, mistral-common,
#     and a custom torch build) frequently clobbers Unsloth's pinned torch/xformers.
# Wall-clock without vLLM on L4 + Qwen 3B is ~55-70 min for the full 14-task run.

# IMPORTANT: do this LAST so nothing above re-upgrades it.
# `transformers` in the unsloth/trl stack pins `huggingface_hub>=0.34,<1.0`.
# Many Colab runtimes ship `huggingface_hub==1.12.0` preinstalled, which breaks
# `from unsloth import FastLanguageModel` at the dep-version check.
# Force a compatible version with --force-reinstall so the existing 1.x is replaced.
!pip install -q --force-reinstall --no-deps "huggingface_hub>=0.34,<1.0"

# After this cell finishes, RESTART THE RUNTIME ONCE, then re-run from Cell 1.
# Reason: if `huggingface_hub` (or a partially-installed `vllm`) was already
# imported by the runtime before these fixes ran, Python has cached the broken
# module objects and the new installs won't take effect until the kernel restarts.
# (On a fresh Colab session this is a no-op.)
print("\nIf this is your FIRST run on a fresh runtime: continue normally.")
print("Otherwise: Runtime -> Restart, then re-run from Cell 1.")

## 2 · Clone the environment repo (Colab)

In [ ]:
import os, subprocess, sys

REPO_URL = os.environ.get(
    "BIORESEARCH_REPO_URL",
    "https://huggingface.co/spaces/anirudhchida/bioresearch",
)

if not os.path.isdir("bioresearch"):
    if "anirudhchida" in REPO_URL:
        raise RuntimeError(
            "Set BIORESEARCH_REPO_URL (or edit REPO_URL above) to your fork — "
            "the placeholder URL will 404 on clone."
        )
    subprocess.run(["git", "clone", REPO_URL, "bioresearch"], check=True)

sys.path.insert(0, "/content/bioresearch")
print("Setup complete")

## 3 · Boot the OpenEnv server

The reward loop talks HTTP to this subprocess. Current OpenEnv releases expose `/health`, `/schema`, `/docs`, and `/openapi.json` — the legacy `/info` is gone.

In [ ]:
import subprocess, time, httpx

server_proc = subprocess.Popen(
    ["uvicorn", "server.app:app", "--host", "127.0.0.1", "--port", "8000"],
    cwd="/content/bioresearch",
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)

for _ in range(40):
    try:
        if httpx.get("http://127.0.0.1:8000/health", timeout=1.0).status_code == 200:
            print("Server up")
            break
    except Exception:
        time.sleep(1.0)
else:
    raise RuntimeError("OpenEnv server failed to start")

## 4 · Wire up `training_core` + Trackio

We reuse [`training_a100.py`](../training_a100.py) verbatim — every helper there (`setup_trackio`, `collect_rollouts_with_pool`, `reward_distribution_diagnostic`, `before_after_table`, `render_sample_transcripts`, `make_lab_shaping_callback`) is GPU-agnostic.

L4-specific lab budget: `TRAIN_LAB_MAX_STEPS=4` — the standard default. Each lab task in the reward fn does up to that many extra `model.generate` calls per completion. With `num_generations=8` and ~57% of prompts being lab tasks, this is the dominant per-step wall-clock contributor; T4 cuts it to 2 (saves time at the cost of process-reward density), L40s pushes it to 6 (uses 48 GB headroom for richer process signal). 4 is the L4 balance point.

Trackio init is wrapped — if the install in Cell 1 failed or the network blocks the HF Space, `setup_trackio` degrades gracefully to `report_to="none"` and the rest of the notebook still produces all the same artifacts (just no live dashboard). Set `HF_SPACE_ID` to a `username/space-name` you own to enable persistent sync.

In [ ]:
import os
import training_core
import training_a100
from training_core import (
    LEGACY_TASKS, LAB_TASKS, ALL_TASKS, DEFAULT_T4_TASKS,
    TRAIN_LAB_MAX_STEPS, EVAL_LAB_MAX_STEPS,
)

# L4 lab rollout budget: 4 (standard default). Each lab task in the reward function
# does up to TRAIN_LAB_MAX_STEPS extra model.generate() calls per completion -- at
# num_generations=8 with ~57% lab tasks per step, that's roughly 8 * 4 = 32 extra
# generates per lab prompt per step. L4's bf16 native + Ada arithmetic throughput
# absorbs this comfortably; T4 caps at 2 to save time, L40s uses 6 for denser
# process signal.
os.environ["TRAIN_LAB_MAX_STEPS"] = "4"
training_core.TRAIN_LAB_MAX_STEPS = 4
training_core.EVAL_LAB_MAX_STEPS = 10

SERVER_URL = "http://127.0.0.1:8000"
training_core.configure_env(SERVER_URL)

# Optional persistent Trackio dashboard. Leave HF_SPACE_ID unset for a
# local-only dashboard (still visible in Colab via the trackio CLI).
trackio_cfg = training_a100.setup_trackio(
    project="bioresearch-grpo-l4",
    run_name="qwen2p5-3b-l4-120steps",
    space_id=os.environ.get("HF_SPACE_ID"),
    config={"model": "Qwen2.5-3B-Instruct-bnb-4bit", "lora_r": 16, "num_generations": 8, "gpu": "L4"},
)
print("Trackio config:", trackio_cfg)

probe = training_core.env_reset(task_type="dna_classification").observation
print(f"Probe OK  task_id={probe.task_id}  question[:60]={probe.question[:60]!r}")

## 5 · Choose the task list

Default trains all 14 tasks — this is the demo run. The 6-task T4 subset and a custom subset are documented as commented lines.

In [ ]:
TASK_LIST = ALL_TASKS                       # 14 reward curves — full demo
# TASK_LIST = DEFAULT_T4_TASKS              # 6-task fast subset for smoke runs
# TASK_LIST = ["dna_reasoning", "clinical_diagnosis", "protein_hypothesis_lab"]

print("Training on:", TASK_LIST)
print("  legacy:", [t for t in TASK_LIST if t in LEGACY_TASKS])
print("  lab   :", [t for t in TASK_LIST if t in LAB_TASKS])

## 6 · Build the mixed-task dataset

`n_per_task=6` (84 rows for all 14 tasks). Half-step between T4's 56 (n=4) and L40s's 112 (n=8). With `pdtbs=2 × ga=2 = 4` prompts/step and `max_steps=120`, that's **~5.7 epochs** — same regime as the T4 (5.7) and L40s (5.4) runs where curves empirically stabilise without overfitting.

In [ ]:
N_PER_TASK = 6
train_ds = training_core.build_mixed_dataset(TASK_LIST, n_per_task=N_PER_TASK, seed=42)
print(train_ds)
print("Sample row keys:", train_ds.column_names)

## 7 · Load Qwen2.5-3B + LoRA (r=16)

Hard-pinned to **Qwen2.5-3B-Instruct-bnb-4bit**. See the rationale in Cell 0 — bigger than T4's 1.5B (closes the knowledge ceiling on `protein_function` / `kegg_pathway_reasoning`), smaller than the colab_pro auto-detect's 7B (~12 GB headroom buys safety margin and avoids the 7B + Unsloth `matmul_lora` dtype edges).

This cell bakes in **all five fixes** from today's `colab_pro` notebook debugging:

1. `dtype=torch.bfloat16` passed explicitly to `from_pretrained` so the 4-bit compute-dtype is pinned (some bnb-4bit checkpoints default to fp16 silently).
2. `use_gradient_checkpointing=True` (HF native) instead of `"unsloth"` — the unsloth checkpointer drops autocast on backward recompute, which produces fp32 inputs to `matmul_lora` while the base layer outputs stay bf16, and the kernel hard-asserts equal dtypes. HF native respects autocast.
3. `model.generation_config.max_length = None` to silence the `~thousands of` "Both max_new_tokens and max_length are set" warnings (Qwen2.5 ships `max_length=32768` baked in; HF compares against our explicit `max_new_tokens` and warns every generate call).
4. Force-cast trainable LoRA params back to `bfloat16` after `get_peft_model` — PEFT's `autocast_adapter_dtype=True` defaults to fp32 promotion. Cell 20 does the same again post-trainer-init (PEFT re-promotes during wrapping).
5. VRAM assertion `> 12 GB free` — Qwen 3B + r=16 LoRA + bf16 ought to leave ~14-16 GB free on a fresh L4 (22.5 GB total). Anything tighter means a deps-stack regression worth catching loudly before training.

In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"          # fixed for L4 demo
# MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"      # smoke test only — DO NOT use for the L4 demo run
# MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"        # use train_grpo_colab_pro.ipynb instead — 7B + L4 needs the auto-detect dtype block
MAX_SEQ_LEN = 2560                                           # 2048 on T4, 3072 on L40s/A100 — 2560 fits L4 with comfortable margin

# L4 = Ada sm_89 = bf16 native. Pin compute dtype so Unsloth's 4-bit loader
# doesn't fall back to fp16 silently (which would defeat half the reason we
# moved up from T4) AND so the matmul_lora kernel sees consistent dtypes
# end-to-end. fp16 fallback would be harmless for correctness but doubles
# gradient noise on the small-batch GRPO updates.
TARGET_DTYPE = torch.bfloat16

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=TARGET_DTYPE,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    # NOTE: HF-native gradient checkpointing (True) instead of "unsloth".
    # The "unsloth" path is faster + uses less VRAM, but it re-runs the forward
    # WITHOUT an autocast scope active. During GRPO backward that produces fp32
    # inputs to Unsloth's matmul_lora kernel while the base layer output stays
    # bf16, which trips:
    #   RuntimeError: self and mat2 must have the same dtype, but got Half and Float
    # HF native (True) honours autocast on recompute, so dtypes stay consistent.
    # Memory cost on L4 with 3B+r=16 is ~1-2 GB — well within the 22.5 GB budget.
    use_gradient_checkpointing=True,
    random_state=42,
)

# Belt-and-suspenders: force every trainable LoRA parameter to TARGET_DTYPE.
# PEFT's `autocast_adapter_dtype=True` default promotes the LoRA A/B matrices
# to fp32 inside `get_peft_model`. Unsloth's matmul_lora kernel does not promote
# inputs; it asserts equal dtypes and crashes if base activations (bf16) and
# LoRA weights (fp32) disagree. Cell 20 does this again post-trainer-init.
_n_cast = 0
for _name, _p in model.named_parameters():
    if _p.requires_grad and _p.dtype == torch.float32:
        _p.data = _p.data.to(TARGET_DTYPE)
        _n_cast += 1
if _n_cast:
    print(f"Cast {_n_cast} fp32 LoRA params -> {TARGET_DTYPE}")

tokenizer.padding_side = "left"
training_core.configure_model(model, tokenizer, max_new_tokens=320)   # 256 on T4 -> 320 on L4 (more rope room for 3B)

# Suppress HF's "Both max_new_tokens and max_length are set" warning at the source.
# Qwen2.5's generation_config.json ships with max_length=32768; we always pass
# max_new_tokens explicitly (320 here, plus TRL's max_completion_length=384 in
# Cell 20). Dropping max_length from the model-level config means HF won't warn
# from ANY generate path - _generate_once (lab rollouts), TRL's GRPO sampler
# (~32 generates/step), or trainer.evaluate. Without this fix the warning spams
# stdout thousands of times over a 120-step run, drowning the actual training logs.
if model.generation_config is not None:
    model.generation_config.max_length = None

if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM headroom after model load: {free / 1e9:.1f} GB free / {total / 1e9:.1f} GB total")
    assert free > 12 * 1e9, (
        f"Only {free / 1e9:.1f} GB free after loading 3B on L4 - expected >12 GB on a fresh 22.5 GB card. "
        "Something in the deps stack regressed; check 4-bit loading + LoRA r=16 + dtype=bf16."
    )
print("Model + tokenizer loaded and registered with training_core")

## 8 · Baseline rollouts (BEFORE training)

Run the *untrained* policy on a fixed seed of 4 held-out `task_id`s per task. We capture the pool here and reuse it after training so the before/after compare on identical briefs.

L40s used n=5 per task, T4 uses n=3; L4 uses n=4 — splits the difference. Each rollout costs ~4-6s on L4 (vs ~8-12s on T4) so n=4 keeps baseline collection under ~5 min while giving slightly tighter per-task means than the T4 run.

In [ ]:
FastLanguageModel.for_inference(model)        # 2x faster greedy-ish gen

baseline_rows, eval_pool = training_a100.collect_rollouts_with_pool(
    TASK_LIST, n_per_task=4, seed=2026,
)
training_a100.save_rollouts(baseline_rows, "baseline_rollouts.json")

# Per-task summary so we have a number to point at on the slide.
import statistics
print(f"Baseline rollouts (n={len(baseline_rows)}):")
for task in TASK_LIST:
    rs = [r["reward"] for r in baseline_rows if r["task_type"] == task]
    if rs:
        print(f"  {task:30s}  mean={statistics.fmean(rs):.3f}  n={len(rs)}")

FastLanguageModel.for_training(model)         # back to training mode

## 9 · Reward-distribution diagnostic

Score a hand-curated bag of completions per task through the reward function. The point isn't to evaluate the model — the completions are canned. The point is to **prove to the judges** that the reward signal:

1. Spans a meaningful range (not all 0 or 1).
2. Reflects completion quality (correct answers get high reward, garbage gets low).
3. Has variance for GRPO's relative advantage to actually optimize against.

In [ ]:
import matplotlib.pyplot as plt

diag_df = training_a100.reward_distribution_diagnostic(TASK_LIST, n_samples_per_task=4)
print(diag_df)

fig, ax = plt.subplots(figsize=(12, 4))
tasks_in_order = [t for t in TASK_LIST if t in set(diag_df["task_type"])]
data = [diag_df[diag_df["task_type"] == t]["reward"].tolist() for t in tasks_in_order]
ax.boxplot(data, labels=tasks_in_order, vert=True, showmeans=True)
ax.set_ylabel("Reward (canned-completion bag)")
ax.set_title("Reward signal has variance — GRPO has something to optimize")
ax.set_ylim(0, 1)
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("reward_diagnostic.png", dpi=140)
plt.show()

## 10 · `GRPOConfig` + `GRPOTrainer` (Trackio-wired)

L4 tunings: `num_generations=8`, `max_completion_length=384`, `max_steps=120`, `save_steps=30`, `bf16=True` (native — no fp16 forcing), cosine LR with 5% warmup. Each dimension is set independently:

- **`num_generations=8`** (vs T4's 4, L40s's 10): doubles T4's advantage signal. GRPO advantage variance scales `1/num_generations`, so going 4 → 8 halves the noise per step. KV cache for 8 generations on a Qwen 3B at max_completion=384 fits in ~3 GB, well within L4's 22.5 GB.
- **`max_completion_length=384`** (vs T4's 256, L40s's 640): 50% larger than T4 to give the 3B model room to articulate the lab-task chain-of-action without overshooting the per-step wall-clock budget.
- **`max_steps=120`** (vs T4's 80, L40s's 150): with 84 prompts × `pdtbs=2 × ga=2 = 4` per step, that's exactly 5.7 epochs — same regime as both sister notebooks where curves stabilise.
- **`save_steps=30`** (vs T4's 20, L40s's 50): checkpoints every ~15 min on L4 wall-clock. Cheap (~75 MB per checkpoint with 3B+r=16) and resilient to mid-run disconnect.
- **`bf16=True`, `fp16=False`** — Ada sm_89 has full bf16 throughput. Halves the gradient-noise contribution from numerical precision on small-batch GRPO updates vs T4's fp16-only path.
- **Cosine LR with `warmup_ratio=0.05`** — adopted from the `colab_pro` notebook. Warmup avoids early-step grad explosions on the first lab-task batches; cosine decay lets the run land softly at step 120 instead of training at peak LR all the way through.

**`USE_VLLM`** auto-detects vLLM and enables colocated rollouts. We deliberately skip the vLLM install in Cell 2 (Unsloth import-hook crash on partial installs), so this will normally print "vLLM not available" and use native HF generation.

**Post-trainer-init dtype reconciliation block** (lines below `trainer = GRPOTrainer(...)`) is the second half of today's `colab_pro` notebook fix: PEFT's `autocast_adapter_dtype=True` re-promotes LoRA params to fp32 *during* `GRPOTrainer.__init__`, so the cast we did in Cell 14 isn't enough on its own. We walk every trainable param after init, force back to bf16, and align `trainer.args.bf16/fp16` flags with what we actually have. Without this the run dies in the first backward pass.

**`LabShapingCallback`** drains the per-step lab rollout log on every `on_log` event so the dashboard shows process vs terminal reward decomposition for each lab task.

In [ ]:
import torch
from trl import GRPOConfig, GRPOTrainer

reward_fns = [training_core.make_reward_fn(t) for t in TASK_LIST]
print("Reward functions wired:")
for fn in reward_fns:
    print(" -", fn.__name__)

# L4 = Ada sm_89 = bf16 native. Hardcode bf16=True; no fp16 forcing like T4.
USE_VLLM = False
try:
    import vllm  # noqa: F401
    USE_VLLM = True
    print("vLLM detected -> enabling fast colocated rollouts")
except Exception:
    print("vLLM not available -> using native HF generation (expected on Colab L4)")

_grpo_kwargs = dict(
    output_dir="grpo_bioresearch_l4",
    learning_rate=3e-6,
    per_device_train_batch_size=2,         # 1 on T4 -> 2 on L4 (more VRAM)
    gradient_accumulation_steps=2,         # 4 on T4 -> 2 on L4 (4 prompts/step effective batch)
    num_generations=8,                     # 4 on T4 -> 8 on L4 (~halve advantage variance)
    max_prompt_length=1280,                # 1024 on T4 -> 1280 on L4
    max_completion_length=384,             # 256 on T4 -> 384 on L4 (3B needs more rope)
    max_steps=120,                         # 80 on T4 -> 120 on L4 (~5.7 epochs at n_per_task=6)
    logging_steps=1,
    save_strategy="steps",
    save_steps=30,                         # 20 on T4 -> 30 on L4 (~15 min checkpoints, ~75 MB each)
    bf16=True,                             # Ada sm_89 native — overrides any T4-style fp16 forcing
    fp16=False,                            # explicit -- never both flags True
    beta=0.04,
    max_grad_norm=1.0,
    # Cosine LR with 5% warmup -- adopted from train_grpo_colab_pro.ipynb.
    # Warmup avoids early-step grad explosions on the first lab-task batches;
    # cosine decay lands the run softly at step 120 instead of training at peak
    # LR all the way through.
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    seed=42,
    use_vllm=USE_VLLM,
    **trackio_cfg,
)
if USE_VLLM:
    _grpo_kwargs["vllm_mode"] = "colocate"  # single-GPU L4

# Forward-compat: older TRL versions don't know `vllm_mode`, newer don't know `use_vllm`.
# Strip unknowns until GRPOConfig accepts the dict.
while True:
    try:
        grpo_config = GRPOConfig(**_grpo_kwargs)
        break
    except TypeError as exc:
        msg = str(exc)
        dropped = False
        for k in ("vllm_mode", "use_vllm"):
            if k in _grpo_kwargs and k in msg:
                print(f"[GRPOConfig] dropping unsupported kwarg {k!r} ({msg.splitlines()[0]})")
                _grpo_kwargs.pop(k)
                dropped = True
                break
        if not dropped:
            raise

# Receipts so the run is reproducible from the notebook output alone.
_dataset_size = len(train_ds)
_prompts_per_step = grpo_config.per_device_train_batch_size * grpo_config.gradient_accumulation_steps
_epochs = (grpo_config.max_steps * _prompts_per_step) / max(_dataset_size, 1)
print(f"\nResolved GRPOConfig (L4):")
print(f"  max_steps              = {grpo_config.max_steps}")
print(f"  num_generations        = {grpo_config.num_generations}")
print(f"  max_completion_length  = {grpo_config.max_completion_length}")
print(f"  max_prompt_length      = {grpo_config.max_prompt_length}")
print(f"  prompts/step           = {_prompts_per_step}  (pdtbs={grpo_config.per_device_train_batch_size} x ga={grpo_config.gradient_accumulation_steps})")
print(f"  dataset size           = {_dataset_size}")
print(f"  -> epochs              = {_epochs:.2f}")
print(f"  lr_scheduler_type      = {grpo_config.lr_scheduler_type}")
print(f"  warmup_ratio           = {grpo_config.warmup_ratio}")
print(f"  save_steps             = {grpo_config.save_steps}")
print(f"  bf16 / fp16            = {grpo_config.bf16} / {grpo_config.fp16}")

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=grpo_config,
    train_dataset=train_ds,
    reward_funcs=reward_fns,
    callbacks=[training_a100.make_lab_shaping_callback()],
)

# ------------------------------------------------------------------
# Post-trainer-init dtype reconciliation.
#
# Two things conspire to cause:
#     RuntimeError: self and mat2 must have the same dtype, but got Half and Float
# inside `unsloth/kernels/utils.py::matmul_lora` on the first GRPO backward:
#
# 1. PEFT defaults to `autocast_adapter_dtype=True`, which promotes the LoRA
#    A/B matrices to fp32 every time the model is (re-)wrapped — including
#    inside `GRPOTrainer.__init__`. So the Cell 14 cast gets undone here.
# 2. Unsloth's matmul_lora kernel does NOT promote inputs; it asserts equal
#    dtypes and crashes if base activations (BFloat16) and LoRA weights
#    (Float) disagree.
#
# The fix is to cast trainable params back to the base-model dtype AFTER the
# trainer has wrapped the model. Idempotent, <50ms, defensive against any
# future TRL/PEFT version that re-introduces the auto-promotion.
# ------------------------------------------------------------------

# Find the actual base activation dtype by looking at a non-trainable weight.
_base_dtype = None
for _n, _p in trainer.model.named_parameters():
    if not _p.requires_grad and _p.dtype in (torch.float16, torch.bfloat16):
        _base_dtype = _p.dtype
        break
if _base_dtype is None:
    # Fallback: trust the GRPOConfig flags.
    _base_dtype = torch.bfloat16 if grpo_config.bf16 else torch.float16
print(f"Base model compute dtype detected as: {_base_dtype}")

# Cast every trainable (LoRA) param to the base dtype.
_n_cast = 0
_n_kept = 0
for _n, _p in trainer.model.named_parameters():
    if not _p.requires_grad:
        continue
    if _p.dtype != _base_dtype:
        _p.data = _p.data.to(_base_dtype)
        _n_cast += 1
    else:
        _n_kept += 1
print(f"Reconciled LoRA dtypes: cast {_n_cast} -> {_base_dtype}, "
      f"kept {_n_kept} already-matching params")

# Align the GRPOConfig precision flags with the actual base dtype, so autocast
# contexts inside GRPOTrainer don't re-introduce the mismatch.
trainer.args.bf16 = (_base_dtype == torch.bfloat16)
trainer.args.fp16 = (_base_dtype == torch.float16)
print(f"Aligned trainer.args: bf16={trainer.args.bf16}  fp16={trainer.args.fp16}")

# Loud sanity check — if base dtype is fp16 on L4, something is wrong (the
# 4-bit loader silently fell back). Surface it instead of letting bf16-targeted
# tunings train at fp16 quality.
if _base_dtype != torch.bfloat16:
    print(f"\n[WARN] Expected bf16 base dtype on L4, got {_base_dtype}. "
          "Run will continue but the bf16-vs-fp16 noise advantage discussed in Cell 19 is lost.\n"
          "Check: (a) GPU is actually L4 (not T4), (b) FastLanguageModel.from_pretrained got dtype=torch.bfloat16.")

print("\nTrainer ready")

## 11 · Train

Expected wall-clock on Colab Pro L4: **~55-70 min train + ~10 min eval = ~65-80 min total** for the full 14-task run.

If the runtime disconnects mid-train, every 30 steps (~15 min) we wrote a LoRA checkpoint to `grpo_bioresearch_l4/checkpoint-XX/` (~75 MB each). Resume in a fresh runtime by re-running Cells 1-20 (which rebuild trainer + dataset deterministically) then in place of `trainer.train()` below run:

```python
import os, glob
ckpts = sorted(
    glob.glob("grpo_bioresearch_l4/checkpoint-*"),
    key=lambda p: int(p.rsplit("-", 1)[1]),
)
print(f"Resuming from {ckpts[-1]}")
trainer.train(resume_from_checkpoint=ckpts[-1])
```

That picks up at the last successful checkpoint instead of restarting from step 0.

In [ ]:
trainer.train()
training_a100.trackio_finish()  # noop if Trackio is not in use

## 12 · Trained rollouts (AFTER training)

Same `task_ids` as Cell 16 — `eval_pool` pins them. This is what makes the before/after table a *paired* comparison instead of two unpaired draws. We use `n=4` here (matching baseline) so every trained row has a baseline counterpart.

In [ ]:
FastLanguageModel.for_inference(model)

trained_rows = training_a100.collect_rollouts(
    TASK_LIST, n_per_task=4, seed=2026, task_id_pool=eval_pool,
)
training_a100.save_rollouts(trained_rows, "trained_rollouts.json")

import statistics
print(f"Trained rollouts (n={len(trained_rows)}):")
for task in TASK_LIST:
    rs = [r["reward"] for r in trained_rows if r["task_type"] == task]
    if rs:
        print(f"  {task:30s}  mean={statistics.fmean(rs):.3f}  n={len(rs)}")

## 13 · Before-vs-after table + bar chart

Headline judging artifact #1. Per-task delta in mean reward, sorted by biggest win first. The bar chart makes the wins obvious at a glance.

**Reading the table on a 3B model:** GRPO teaches *behavior* (JSON discipline, phase transitions, tool-call selection), not *facts*. The 3B base closes **most** of the knowledge ceiling that limits the T4 / 1.5B run on `protein_function` and `kegg_pathway_reasoning` — expect visible green deltas on those tasks where T4 shows flat. Format- and procedure-bound tasks (`dna_classification`, `evidence_ranking`, `clinical_diagnosis`, all the `*_lab` tasks) should still show the largest deltas because GRPO compounds fastest where the reward signal gates on observable behavior.

If a particular task is still flat after this run, the next-best lever is the **7B** model (`train_grpo_colab_pro.ipynb`); switching to a bigger Qwen tier moves the knowledge-ceiling needle far more than burning more steps at 3B.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

table = training_a100.before_after_table(baseline_rows, trained_rows)
print(table.to_string(index=False) if hasattr(table, "to_string") else table)

if hasattr(table, "iloc"):
    tasks = table["task_type"].tolist()
    deltas = table["delta"].tolist()
    baselines = table["baseline_mean"].tolist()
    traineds = table["trained_mean"].tolist()
else:
    tasks = [r["task_type"] for r in table]
    deltas = [r["delta"] for r in table]
    baselines = [r["baseline_mean"] for r in table]
    traineds = [r["trained_mean"] for r in table]

x = np.arange(len(tasks))
fig, ax = plt.subplots(figsize=(13, 5))
width = 0.38
ax.bar(x - width/2, baselines, width, label="Baseline (untrained)")
ax.bar(x + width/2, traineds, width, label="After GRPO training")
for i, d in enumerate(deltas):
    color = "green" if d > 0 else ("red" if d < 0 else "gray")
    ax.annotate(f"{d:+.2f}", xy=(x[i], max(baselines[i], traineds[i]) + 0.02),
                ha="center", color=color, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(tasks, rotation=40, ha="right")
ax.set_ylabel("Mean reward")
ax.set_title("Before vs After GRPO — per-task mean reward (paired task_ids, Colab Pro L4 + Qwen2.5-3B)")
ax.set_ylim(0, 1)
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("before_after_bar.png", dpi=140)
plt.show()

## 14 · Sample rollout transcripts (the headline demo artifact)

Headline judging artifact #2. For each task, the 2 `task_id`s with the largest positive delta are shown side-by-side: prompt → baseline completion + reward → trained completion + reward. Judges can literally read the qualitative jump.

In [ ]:
from IPython.display import Markdown, display

transcripts_md = training_a100.render_sample_transcripts(
    baseline_rows, trained_rows, k=2, max_chars_per_block=600,
)
with open("sample_transcripts.md", "w") as f:
    f.write(transcripts_md)
print(f"Wrote sample_transcripts.md ({len(transcripts_md)} chars)")

display(Markdown(transcripts_md))

## 15 · Per-task reward curves (from `trainer.state.log_history`)

Live Trackio is the primary instrument; this is the static fallback so the artifact survives the runtime ending. Each curve is a single task's reward function — `0.0` rows (rows from other tasks) are masked before smoothing.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log_df = pd.DataFrame(trainer.state.log_history)
print("Reward-related columns:", [c for c in log_df.columns if "reward" in c])

fig, ax = plt.subplots(figsize=(12, 5))
for task in TASK_LIST:
    candidates = [c for c in log_df.columns
                  if task in c and "reward" in c and "std" not in c.lower() and "process" not in c and "terminal" not in c]
    if not candidates:
        continue
    col = candidates[0]
    subset = log_df[["step", col]].dropna()
    subset = subset[subset[col] > 0.0].reset_index(drop=True)
    if subset.empty:
        continue
    smooth = subset[col].rolling(window=10, min_periods=1).mean()
    ax.plot(subset["step"], smooth, linewidth=2.0, label=task)

ax.set_xlabel("GRPO step")
ax.set_ylabel("Reward (10-step EMA, gated rows only)")
ax.set_title(f"Per-task reward curves — Colab Pro L4 (Qwen2.5-3B, num_generations={grpo_config.num_generations}, max_steps={grpo_config.max_steps})")
ax.legend(loc="best", fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("training_curves.png", dpi=140)
plt.show()

## 16 · Save LoRA + (optional) Hub push + teardown

In [ ]:
model.save_pretrained("grpo_bioresearch_l4_lora")
tokenizer.save_pretrained("grpo_bioresearch_l4_lora")
print("LoRA saved to grpo_bioresearch_l4_lora/  (~75 MB checkpoint - easy to share)")

# Optional: push to the Hub if HF_TOKEN is set in the env. The 3B + r=16 LoRA
# weighs in around 75 MB so the push completes in well under a minute and the
# resulting repo is cheap to pull from inference notebooks (see
# train_grpo_inference_mirror.ipynb).
HF_REPO = os.environ.get("HF_PUSH_REPO")
if HF_REPO and os.environ.get("HF_TOKEN"):
    try:
        model.push_to_hub(HF_REPO, token=os.environ["HF_TOKEN"])
        tokenizer.push_to_hub(HF_REPO, token=os.environ["HF_TOKEN"])
        print(f"Pushed to https://huggingface.co/{HF_REPO}")
    except Exception as e:
        print(f"Hub push failed ({type(e).__name__}: {e}); LoRA still saved locally.")
else:
    print("Skipped Hub push (set HF_PUSH_REPO + HF_TOKEN to enable).")

server_proc.terminate()
try:
    server_proc.wait(timeout=5)
except Exception:
    server_proc.kill()
print("Server terminated")